# Evolving k-Threshold Secret Sharing

Evolving secret sharing is a scheme designed for environments where the set of participants is not fixed in advance.

In Shamir's threshold secret sharing, the number of participants n must be decided before generating shares. In contrast, the Evolving scheme assumes participants join incrementally over time, requiring no prior commitment to n.

When a new participant joins, the dealer issues a share only to that participant. No communication with existing participants or redistribution of existing shares is required.

---

## Threshold k

k or more shares are required to recover the secret.

### k − 1 or fewer

The secret cannot be recovered from k − 1 or fewer shares.

---

## Properties of the Evolving Scheme

- No need to fix the number of participants in advance
- Participants can be added incrementally
- Only the new participant receives a new share
- Existing shares are never modified
- No re-communication with existing participants is required
- The scheme flexibly accommodates expansion of the participant set

---

## Difference from Shamir's Scheme

In Shamir's threshold secret sharing, the number of participants n must be fixed before generating shares.

In contrast, the Evolving scheme assumes the participant set may expand during operation. Therefore, even when new participants are added, the scheme continues without any communication with existing participants or redistribution of existing shares.

---

## Adding Participants in This Program

When a new participant arrives:

1. Assign a new participant identifier
2. Generate a share for that participant
3. Distribute the share to the new participant

Existing shares are retained as-is, and the secret remains unchanged.

---

## Share Size

In the Evolving scheme, share size generally differs per participant.

The share size of participant t is expressed as:

\[
\log |X_t|
\]

where $X_t$ is the share space assigned to participant $t$.

In this program, the share size of each participant is computed, allowing observation of how share size changes as the number of participants increases.

---

---

## Implementation

Below is a simplified implementation of the Evolving 2-threshold secret sharing scheme.

This implementation uses XOR operations to provide an intuitive understanding of the secret sharing structure.

Rather than polynomial computation over a finite field as in Shamir's scheme, the Evolving behavior is reproduced using random values and XOR operations.

In the code, shares are extended at each time step t, allowing new participants to be added while retaining existing information.

The following assumes that participants 1 and 2 are present initially.

In [1]:
import secrets

# =====================================================
# XOR function
# Applies bitwise XOR to two byte sequences
# Example: a=0b1010, b=0b1100 → 0b0110
# The fundamental operation used for encryption and decryption
# =====================================================
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))


# =====================================================
# Evolving 2-Threshold Secret Sharing
#
# [Idea]
#   - Split secret S into shares and distribute to participants
#   - 2 or more participants can recover the secret (1 alone cannot)
#   - New participants can be added at each time step t ("Evolving")
#
# [What each participant holds]
#   - Rt      : A random value unique to this participant
#   - Ri ⊕ S : XOR of an earlier participant's random value Ri and secret S
#
# [Recovery]
#   - XOR participant i's Ri with another participant's Ri ⊕ S to reveal S
#     Ri ⊕ (Ri ⊕ S) = S  ← XOR property: the same value XORed twice cancels out
# =====================================================
class EvolvingSecretSharing:
    def __init__(self, secret):
        self.secret = secret.encode()   # Convert secret to bytes
        self.random_values = []         # Store each participant's random value Rt in order
        self.shares = []                # Store each participant's share (Rt + encrypted secret)

    # =====================================================
    # Generate share for time step t
    # Call this each time a new participant joins
    # =====================================================
    def evolve(self):
        # Generate a random value for the new participant (same length as secret)
        Rt = secrets.token_bytes(len(self.secret))
        self.random_values.append(Rt)

        # Current participant number (= total number of participants so far)
        t = len(self.random_values)

        # Compute XOR of each previous participant's Ri with secret S
        # → By receiving Ri ⊕ S, the new participant can recover the secret
        #   by combining with any earlier participant
        encrypted_parts = []
        for Ri in self.random_values[:-1]:  # Exclude own Rt
            encrypted = xor_bytes(Ri, self.secret)
            encrypted_parts.append(encrypted)

        # Store the share as a dictionary
        Xt = {
            "time": t,                          # Participant number (time step)
            "Rt": Rt,                           # This participant's random value
            "encrypted_parts": encrypted_parts  # Data for combining with earlier participants
        }
        self.shares.append(Xt)
        return Xt

    # =====================================================
    # Recover secret (with threshold check)
    #
    # Args:
    #   Ri                : Random value of an existing participant
    #   encrypted         : Ri ⊕ S held by another participant
    #   participants_count: Number of participants present for recovery
    #   threshold         : Minimum number of participants required (default 2)
    # =====================================================
    def recover_secret(self, Ri, encrypted, participants_count, threshold=2):
        # Reject recovery if below threshold (important for security)
        if participants_count < threshold:
            print(f"[ERROR] Participant count {participants_count} is below threshold {threshold}. Recovery denied.")
            return False

        # Use the property Ri ⊕ (Ri ⊕ S) = S to extract the secret
        recovered = xor_bytes(Ri, encrypted)
        return recovered.decode()  # Convert bytes back to string


# =====================================================
# Display utility
# Print share contents in a readable format
# =====================================================
def print_share(share):
    t = share['time']
    print(f"\n=== Time t={t} ===")
    print(f"R{t}:")                    # This participant's random value (hex)
    print(share["Rt"].hex())
    print("\nRi ⊕ S:")
    if not share["encrypted_parts"]:
        # t=1: No earlier participants yet, so no encrypted data
        print("(not yet available)")
    else:
        # t=2 onwards: Show Ri ⊕ S for each earlier participant
        for i, part in enumerate(share["encrypted_parts"], start=1):
            print(f"R{i} ⊕ S = {part.hex()}")


# =====================================================
# Example: Generate shares for 2 participants
# =====================================================
secret = "HELLO"
ess = EvolvingSecretSharing(secret)

# Participant 1 joins (t=1)
# → Rt only. No earlier participants, so encrypted_parts is empty
share1 = ess.evolve()
print_share(share1)

# Participant 2 joins (t=2)
# → R2 and "R1 ⊕ S" as combination data with participant 1
share2 = ess.evolve()
print_share(share2)


# =====================================================
# Secret Recovery
#
# Steps for participants 1 and 2 to recover the secret:
#   1. Participant 1 provides their random value R1
#   2. XOR R1 with "R1 ⊕ S" held by participant 2
#   3. R1 ⊕ (R1 ⊕ S) = S → secret is revealed
# =====================================================
R1 = ess.random_values[0]                 # Participant 1's random value R1
encrypted = share2["encrypted_parts"][0]  # R1 ⊕ S held by participant 2

# Number of participants present (change to 1 to see recovery failure)
participants_count = 2

recovered = ess.recover_secret(R1, encrypted, participants_count)

print("\n=== Secret Recovery ===")
if recovered is False:
    print("Recovery failed: not enough participants")
else:
    print(f"Participant 1's share R1: {R1.hex()} and")
    print(f"Participant 2's share R1 ⊕ S = {encrypted.hex()} used for recovery")
    print(f"Secret: {recovered}")


=== Time t=1 ===
R1:
2ee0d52486

Ri ⊕ S:
(not yet available)

=== Time t=2 ===
R2:
1cfec0e34b

Ri ⊕ S:
R1 ⊕ S = 66a59968c9

=== Secret Recovery ===
Participant 1's share R1: 2ee0d52486 and
Participant 2's share R1 ⊕ S = 66a59968c9 used for recovery
Secret: HELLO


## Function to Generate Shares for New Participants

In [2]:
# =====================================================
# Function to Add New Participants
#
# [Role]
#   Add add_n new participants to the existing
#   ess (secret sharing object)
#
# [What each new participant receives]
#   - Rt      : A random value unique to this participant
#   - Ri ⊕ S : XOR of each existing participant's random value and secret S
#               → Can recover the secret by combining with any existing participant
#
# Args:
#   ess   : An instance of EvolvingSecretSharing
#   add_n : Number of participants to add
# =====================================================
def add_participants(ess, add_n):
    new_participants = []

    # Record the current number of participants before adding
    # → Assign sequential IDs to new participants starting from here
    # Example: if 2 participants exist, new ones start from ID 3
    current_n = len(ess.random_values)

    for i in range(add_n):

        # Generate a random value for the new participant
        # Must be the same length as the secret (required for XOR)
        Rt = secrets.token_bytes(len(ess.secret))
        ess.random_values.append(Rt)

        # Participant ID (1-indexed)
        participant_id = current_n + i + 1

        # Compute XOR of each existing participant's Ri with secret S
        # → Holding this value allows recovery of the secret
        #   together with the corresponding existing participant
        #   Recovery formula: Ri ⊕ (Ri ⊕ S) = S
        encrypted_parts = []
        for Ri in ess.random_values[:-1]:  # Exclude own Rt
            encrypted = xor_bytes(Ri, ess.secret)
            encrypted_parts.append(encrypted)

        # Store the share as a dictionary
        Xt = {
            "participant_id": participant_id,  # Participant ID
            "Rt": Rt,                          # This participant's random value
            "encrypted_parts": encrypted_parts # Data for combining with existing participants
        }

        # Append to ess.shares so that subsequent new participants
        # can also use this participant's random value
        ess.shares.append(Xt)
        new_participants.append(Xt)

    return new_participants  # Return the list of added participants

## Adding Participants in Practice

In [3]:
import random

# -------------------------
# Number of participants to add
# -------------------------
add_n = 3


# -------------------------
# Initialize participants
# -------------------------
# [Reset] As-is, numbering restarts from 3 every run
# [Carry over] Comment out the line below to continue
#              numbering from the previous run
ess = EvolvingSecretSharing(secret)  # ← Comment out this line to carry over state
share1 = ess.evolve()  # Generate share for participant 1
share2 = ess.evolve()  # Generate share for participant 2


# -------------------------
# Generate new participants
# -------------------------
# Generate add_n shares and add them to ess
# Returns a list of the added participants
new_participants = add_participants(ess, add_n)

print("=== New Participants ===")
for p in new_participants:
    pid = p.get("time") or p.get("participant_id")
    print(f"\nParticipant {pid}")
    print(f"R{pid}: {p['Rt'].hex()}")
    if not p["encrypted_parts"]:
        print("Ri ⊕ S: (not yet available)")
    else:
        # Show Ri ⊕ S for each existing participant
        # → XOR this value with the corresponding Ri to recover the secret
        for i, part in enumerate(p["encrypted_parts"], start=1):
            print(f"R{i} ⊕ S = {part.hex()}")


# -------------------------
# Build full participant list
# -------------------------
# Combine participants added via evolve() (1 and 2)
# with those added via add_participants()
all_participants = ess.shares + new_participants


# -------------------------
# Select participants for recovery
# -------------------------
# Randomly select participants_count people from all participants
#
# Threshold is 2, so 2 or more are needed for recovery
# Change to 1 to see recovery failure
participants_count = 3

selected = random.sample(
    all_participants,
    min(participants_count, len(all_participants))
)

print("\n=== Participants Selected for Recovery ===")
for p in selected:
    pid = p.get("time") or p.get("participant_id")
    print(f"Participant {pid}")


# -------------------------
# Secret recovery
# -------------------------
# Find a valid pair among selected and recover the secret
#
# Valid pair condition:
#   p_b holds "R{pid_a} ⊕ S" for p_a
#   meaning p_b joined after p_a
#
# Recovery formula: Ri ⊕ (Ri ⊕ S) = S
if participants_count < 2:
    print("\n=== Secret Recovery ===")
    print("Recovery failed: not enough participants")
else:
    recovered = False
    for p_a in selected:
        pid_a = p_a.get("time") or p_a.get("participant_id")
        for p_b in selected:
            if p_a is p_b:
                continue
            # Check if p_b holds Ri ⊕ S for p_a
            # → p_b holds it if len(encrypted_parts) >= pid_a
            if pid_a - 1 < len(p_b["encrypted_parts"]):
                Ri        = ess.random_values[pid_a - 1]  # p_a's random value
                encrypted = p_b["encrypted_parts"][pid_a - 1]  # Ri ⊕ S held by p_b
                pid_b     = p_b.get("time") or p_b.get("participant_id")
                recovered = ess.recover_secret(Ri, encrypted, participants_count)

                print("\n=== Secret Recovery ===")
                print(f"Participant {pid_a}'s share R{pid_a}: {Ri.hex()} and")
                print(f"Participant {pid_b}'s share R{pid_a} ⊕ S = {encrypted.hex()} used for recovery")
                print(f"Secret: {recovered}")
                break
        if recovered:
            break

=== New Participants ===

Participant 3
R3: 02ca095a14
R1 ⊕ S = 2e4f447040
R2 ⊕ S = f64c066b14

Participant 4
R4: 69266f3256
R1 ⊕ S = 2e4f447040
R2 ⊕ S = f64c066b14
R3 ⊕ S = 4a8f45165b

Participant 5
R5: 9693f61d6a
R1 ⊕ S = 2e4f447040
R2 ⊕ S = f64c066b14
R3 ⊕ S = 4a8f45165b
R4 ⊕ S = 2163237e19

=== Participants Selected for Recovery ===
Participant 3
Participant 1
Participant 5

=== Secret Recovery ===
Participant 3's share R3: 02ca095a14 and
Participant 5's share R3 ⊕ S = 4a8f45165b used for recovery
Secret: HELLO
